In [ ]:
import numpy as np
from freshflow.loader import load_sales
from freshflow.metrics import build_metrics
from freshflow.scenarios import (
    discount_bands, discount_relationships, baseline_totals, actual_results,
    scenario_table, breakeven_quantity_change,
)

clean = load_sales("../data/processed/freshflow_clean.csv", expected_rows=None)
metrics = build_metrics(clean)

In [ ]:
bands = discount_bands(clean, metrics)

print(f"{'band':12} {'trans':>6} {'units':>7} {'avg qty':>8} {'revenue':>11} {'profit':>11} {'margin%':>8}")
for i, label in enumerate(bands["labels"]):
    print(f"{label:12} {bands['transactions'][i]:6} {bands['units'][i]:7,.0f} {bands['mean_quantity'][i]:8.3f} "
          f"{bands['revenue'][i]:11,.0f} {bands['profit'][i]:11,.0f} {bands['margin_pct'][i]:8.2f}")

In [ ]:
for name, value in discount_relationships(clean, metrics).items():
    print(f"{name:34} {value:.4f}")

In [ ]:
totals = baseline_totals(metrics)
actual_revenue, actual_profit = actual_results(totals)

print("Rows used in every scenario:", totals["rows"])
print("Gross sales:", round(totals["gross"], 2))
print("COGS:", round(totals["cogs"], 2))
print("Actual discount:", round(totals["discount"], 2))
print("Weighted discount %:", round(totals["discount"] / totals["gross"] * 100, 2))
print("Actual net revenue:", round(actual_revenue, 2))
print("Actual gross profit:", round(actual_profit, 2))

In [ ]:
rates = [0, 0.05, 0.10, 0.15, 0.20]
table = scenario_table(totals, rates, [0.0, 0.01, 0.02])

print("SCENARIOS, NOT FORECASTS")
print("response  rate  qty factor     revenue     profit  margin%  profit change  change%")
for row in table:
    print(f"{row['response']:8.2f}  {row['rate']:4.0%}  {row['quantity_factor']:10.3f}  {row['net_revenue']:10,.0f}  "
          f"{row['gross_profit']:9,.0f}  {row['margin_pct']:7.2f}  {row['profit_change']:13,.0f}  {row['profit_change_pct']:7.2f}")

In [ ]:
for rate in rates:
    change = breakeven_quantity_change(totals, rate)
    print(f"Discount {rate:.0%}: quantity must change by {change:.2f}% to keep today's profit")